In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import re
from textblob import TextBlob
from wordcloud import WordCloud
import cufflinks as cf
import os
from google.colab import files
import plotly.express as px

import warnings
warnings.filterwarnings("ignore")
warnings.warn("This will not show")


In [ ]:
df = pd.read_csv("Reviews.csv", engine='python', on_bad_lines='warn')

In [ ]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [ ]:
df.isnull().sum()

,0
Id,0
ProductId,0
UserId,0
ProfileName,1
HelpfulnessNumerator,0
HelpfulnessDenominator,0
Score,0
Time,0
Summary,0
Text,0


In [ ]:
df1 = df.sort_values('ProductId', axis=0, ascending=True, inplace=False,
                     kind='quicksort', na_position='last')

In [ ]:
df2=df1.drop_duplicates(subset={"UserId", "ProfileName", "Time", "Text"}, keep='first', inplace=False)

In [ ]:
df3=df2[df2.HelpfulnessNumerator <= df2.HelpfulnessDenominator]

In [ ]:
df3['Score'].value_counts()

,count
Score,
5,12243
4,2748
1,1770
3,1572
2,1078


In [ ]:
df3.isnull().sum()

,0
Id,0
ProductId,0
UserId,0
ProfileName,1
HelpfulnessNumerator,0
HelpfulnessDenominator,0
Score,0
Time,0
Summary,0
Text,0


In [ ]:
df4 = df3.dropna(subset=["ProfileName", "Summary"])

In [ ]:
df4["ReviewLength"]=df4["Text"].apply(len)
df4["HelpfulnessRatio"]=((df4["HelpfulnessNumerator"]/df4["HelpfulnessDenominator"]).replace(0,np.nan))

In [ ]:
df4["ReviewDate"] = pd.to_datetime(df4["Time"], unit="s")
df4["Year"] = df4["ReviewDate"].dt.year
df4["Month"] = df4["ReviewDate"].dt.month
df4["YearMonth"] = df4["ReviewDate"].dt.to_period("M").astype(str)

In [ ]:
df4.drop(columns=["Time"], inplace=True)
df4.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,ReviewLength,HelpfulnessRatio,ReviewDate,Year,Month,YearMonth
2774,2775,B00002NCJC,A13RRPGE79XFFH,reader48,0,0,5,Flies Begone,We have used the Victor fly bait for 3 seasons...,79,NaN,2010-08-06,2010,8,2010-08
2773,2774,B00002NCJC,A196AJHU9EASJN,Alex Chaffee,0,0,4,thirty bucks?,Why is this $[...] when the same product is av...,266,NaN,2010-08-28,2010,8,2010-08
1243,1244,B00002Z754,A3B8RCEI0FXFI6,B G Chase,10,10,5,WOW Make your own 'slickers' !,I just received my shipment and could hardly w...,475,1.0,2000-06-29,2000,6,2000-06
1244,1245,B00002Z754,A29Z5PI9BW2PU3,Robbie,7,7,5,Great Product,This was a really good idea and the final prod...,168,1.0,2000-06-23,2000,6,2000-06
9526,9527,B00005V3DC,A8KY7S48EW7LW,"A. Daly ""AD""",0,0,5,The Best Cleansing Tea I've Ever Had,I've used this brand for years. If you're feel...,370,NaN,2012-10-14,2012,10,2012-10


In [ ]:
reviews = df4.copy()

In [ ]:
reviews.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,ReviewLength,HelpfulnessRatio,ReviewDate,Year,Month,YearMonth
2774,2775,B00002NCJC,A13RRPGE79XFFH,reader48,0,0,5,Flies Begone,We have used the Victor fly bait for 3 seasons...,79,NaN,2010-08-06,2010,8,2010-08
2773,2774,B00002NCJC,A196AJHU9EASJN,Alex Chaffee,0,0,4,thirty bucks?,Why is this $[...] when the same product is av...,266,NaN,2010-08-28,2010,8,2010-08
1243,1244,B00002Z754,A3B8RCEI0FXFI6,B G Chase,10,10,5,WOW Make your own 'slickers' !,I just received my shipment and could hardly w...,475,1.0,2000-06-29,2000,6,2000-06
1244,1245,B00002Z754,A29Z5PI9BW2PU3,Robbie,7,7,5,Great Product,This was a really good idea and the final prod...,168,1.0,2000-06-23,2000,6,2000-06
9526,9527,B00005V3DC,A8KY7S48EW7LW,"A. Daly ""AD""",0,0,5,The Best Cleansing Tea I've Ever Had,I've used this brand for years. If you're feel...,370,NaN,2012-10-14,2012,10,2012-10


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 102.7 MB/s  0:00:00


In [ ]:
from tqdm import tqdm
from bs4 import BeautifulSoup
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def decontracted(phrase):
    # specific
    phrase = re.sub(r"won't", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not", phrase)

    # general
    phrase = re.sub(r"n\'t", " not", phrase)
    phrase = re.sub(r"\'re", " are", phrase)
    phrase = re.sub(r"\'s", " is", phrase)
    phrase = re.sub(r"\'d", " would", phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t", " not", phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m", " am", phrase)
    return phrase

In [ ]:
STOPWORDS = set(stopwords.words('english'))

# Keep important negation words for sentiment analysis
NEGATION_WORDS = {'no', 'not', 'nor', 'never'}
STOPWORDS = STOPWORDS - NEGATION_WORDS


def preprocess_text(text):

    # Handle missing values
    text = str(text)

    # Remove HTML tags
    text = BeautifulSoup(text, 'lxml').get_text()

    # Remove URLs
    text = re.sub(r'http\S+', '', text)

    # Convert to lowercase
    text = text.lower()

    # Decontract words
    text = decontracted(text)

    # Keep only alphabets and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra spaces
    text = ' '.join(text.split())

    # Remove stopwords but keep negation words
    words = text.split()

    words = [
        word for word in words
        if word not in STOPWORDS
    ]

    return ' '.join(words)


reviews['clean_text'] = reviews['Text'].apply(preprocess_text)

In [ ]:
reviews = reviews[reviews['clean_text'].str.strip() != '']

In [ ]:
reviews['sentiment'] = reviews['Score'].apply(
    lambda x: 0 if x <= 2 else 1
)

In [ ]:
def sentiment_label(score):
    if score <= 2:
        return 0       # Negative
    else:
        return 1       # Positive

reviews['sentiment'] = reviews['Score'].apply(sentiment_label)

In [ ]:
print(reviews['Score'].value_counts().sort_index())

Score
1     1770
2     1078
3     1572
4     2748
5    12242
Name: count, dtype: int64


In [ ]:
print(reviews['sentiment'].value_counts().sort_index())

sentiment
0     2848
1    16562
Name: count, dtype: int64


In [ ]:
X = reviews['clean_text']
y = reviews['sentiment']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=10
)

In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)

Training shape: (15528, 9811)
Testing shape: (3882, 9811)


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

# **Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    C=1,
    max_iter=1000,
    class_weight='balanced'
)

lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

In [ ]:
print("LR expects:", lr.n_features_in_)
print("TF-IDF features:", X_train_tfidf.shape[1])

LR expects: 9811
TF-IDF features: 9811


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    y_test,
    y_pred_lr,
    target_names=['Negative', 'Positive']
))

print(confusion_matrix(y_test, y_pred_lr))

              precision    recall  f1-score   support

    Negative       0.57      0.81      0.67       570
    Positive       0.96      0.89      0.93      3312

    accuracy                           0.88      3882
   macro avg       0.77      0.85      0.80      3882
weighted avg       0.91      0.88      0.89      3882

[[ 462  108]
 [ 351 2961]]


In [ ]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average='macro')

print("Tuned Logistic Regression Accuracy:", accuracy_lr)
print("Tuned Logistic Regression Macro F1:", f1_lr)

Tuned Logistic Regression Accuracy: 0.8817619783616693
Tuned Logistic Regression Macro F1: 0.7980902496259741


## **Naive Bayes**

In [ ]:
from sklearn.naive_bayes import MultinomialNB
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)

In [ ]:
accuracy_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb, average='macro')
print("Naive Bayes Accuracy:", accuracy_nb)
print("Naive Bayes macro F1:", f1_nb)

Naive Bayes Accuracy: 0.8693972179289027
Naive Bayes macro F1: 0.568935243721696


In [ ]:
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

           0       0.94      0.12      0.21       570
           1       0.87      1.00      0.93      3312

    accuracy                           0.87      3882
   macro avg       0.91      0.56      0.57      3882
weighted avg       0.88      0.87      0.82      3882



## **Support Vector Machine (SVM)**

In [ ]:
from sklearn.svm import LinearSVC
svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)

In [ ]:
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Negative", "Positive"]
))

              precision    recall  f1-score   support

    Negative       0.61      0.73      0.67       570
    Positive       0.95      0.92      0.94      3312

    accuracy                           0.89      3882
   macro avg       0.78      0.83      0.80      3882
weighted avg       0.90      0.89      0.90      3882



In [ ]:
accuracy_svm = accuracy_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm, average='macro')
print("Linear SVM Accuracy:", accuracy_svm)
print("Linear SVM F1:", f1_svm)

Linear SVM Accuracy: 0.8923235445646573
Linear SVM F1: 0.8012288786482334


# Comparison

In [ ]:
f1_lr = f1_score(y_test, y_pred_lr, average='macro')
f1_nb = f1_score(y_test, y_pred_nb, average='macro')
f1_svm = f1_score(y_test, y_pred_svm, average='macro')

results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Naive Bayes',
        'Linear SVM'
    ],
    'Accuracy': [
        accuracy_lr,
        accuracy_nb,
        accuracy_svm
    ],
    'Macro F1': [
        f1_lr,
        f1_nb,
        f1_svm
    ]
})

print(results)

                 Model  Accuracy  Macro F1
0  Logistic Regression  0.881762  0.798090
1          Naive Bayes  0.869397  0.568935
2           Linear SVM  0.892324  0.801229


# **Prediction**

In [ ]:
new_reviews = [
    "This product was excellent and I really enjoyed it.",
    "The food was terrible and tasted awful.",
    "The product was okay, nothing special."
]

new_reviews_clean = [
    preprocess_text(review)
    for review in new_reviews
]

In [ ]:
new_reviews_tfidf = tfidf.transform(new_reviews_clean)
predictions = svm_model.predict(new_reviews_tfidf)
print(predictions)

[1 0 1]


In [ ]:
sentiment_names = {
    0: "Negative",
    1: "Positive"
}

for review, prediction in zip(new_reviews, predictions):
    print("Review:", review)
    print("Predicted Sentiment:", sentiment_names[prediction])
    print()

Review: This product was excellent and I really enjoyed it.
Predicted Sentiment: Positive

Review: The food was terrible and tasted awful.
Predicted Sentiment: Negative

Review: The product was okay, nothing special.
Predicted Sentiment: Positive



In [ ]:
import joblib
import sklearn

# Save the exact fitted vectorizer and trained classifier.
artifacts = {
    "tfidf": tfidf,
    "model": svm_model
}

joblib.dump(artifacts, "sentiment_model.joblib")

print("Model and vectorizer saved successfully.")
print("Scikit-learn version:", sklearn.__version__)

In [ ]:
from google.colab import files

files.download("sentiment_model.joblib")